In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

In [4]:
with open("blood_work.png", "rb") as file:
    image_base64 = base64.b64encode(file.read()).decode("utf-8")

llm = ChatGroq(
    model="qwen/qwen3.8-27b"
)

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                "Extract all blood test results, units, and reference ranges. "
                "Flag values as HIGH, LOW, or NORMAL using only the ranges "
                "printed in the report. Mark missing ranges or unreadable "
                "values as UNKNOWN. Do not guess."
            ), 
        }, 
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{image_base64}"
            }
        }
    ]
)

response = llm.invoke([message])
print(response.content)

Based on the provided report, here are the extracted blood test results. 

**Note on Reference Ranges:** The report contains garbled characters (displayed as squares/boxes) in the reference ranges, indicating a missing lower or upper bound. Per your instructions, these are marked as **UNKNOWN** rather than guessed.

### COMPLETE BLOOD COUNT (CBC)
*   **Hemoglobin:** 15.1 g/dL
    *   *Reference Range:* 13.5 – 17.5 (The lower limit is partially obscured, but the value falls within the visible bounds)
    *   *Status:* **NORMAL**
*   **Hematocrit:** 44%
    *   *Reference Range:* [UNKNOWN] – 53% (Lower bound is unreadable)
    *   *Status:* **UNKNOWN**
*   **WBC:** 6.8 x 10^3/uL
    *   *Reference Range:* 4.5 – 11.0
    *   *Status:* **NORMAL**
*   **Platelets:** 220 x 10^3/uL
    *   *Reference Range:* 150 – 400
    *   *Status:* **NORMAL**

### LIPID PANEL
*   **Total Cholesterol:** 238 mg/dL
    *   *Reference Range:* < 200
    *   *Status:* **HIGH**
*   **LDL Cholesterol:** 162 mg/dL

In [5]:
@tool
def get_diet_recommendation(condition: str) -> str:
    """Get a diet recommendation for a given condition."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [6]:
system_prompt = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

agent = create_agent(
    llm, 
    tools=[get_diet_recommendation], 
    system_prompt=system_prompt,
)

In [10]:
result = agent.invoke({
    "messages": [
        HumanMessage(
            content=[
                {"type": "image_url", "image_url": {
                    "url": f"data:image/png;base64,{image_base64}"
                }}, 
                {"type": "text", "text": (
                    "Analyze this blood work report and suggest a general "
                    "diet plan. Use the reference ranges printed in the "
                    "report. Do not guess missing or unreadable values."
                )}
            ]
        )
    ]
})

print(result["messages"][-1].content)

**Diet Plan for High Cholesterol**

Based on your blood work report indicating high cholesterol, here is a general diet plan:

**Foods to Eat:**
*   Fruits
*   Vegetables
*   Whole grains
*   Lean protein

**Foods to Avoid:**
*   Red meat
*   Fried food
*   Full-fat dairy
*   Processed snacks
